In [1]:
import pandas as pd
import numpy as np

In [2]:
def read_data(path , sheet_name=None):
    xls = pd.ExcelFile(path)
    # Read the sheet into a DataFrame
    sheet_data = pd.read_excel(xls, sheet_name=sheet_name, header=None)
    # Extract relevant columns (assumed to have certain keywords, could vary in real sheets)
    # Here we use placeholder for the actual extraction process
    relevant_data = sheet_data.copy()  # This is where extraction logic will be applied
    return xls, relevant_data

In [3]:
# Đọc dữ liệu từ file Excel
file_path = './Metadata/GGA-metal/File excel/Excel-GGA-metal.xlsx'  # Thay bằng đường dẫn thực tế đến file Excel của bạn
xls, sheets = read_data(file_path)


In [4]:
lst_sheets = xls.sheet_names

In [5]:
def find_sample(sheet_name, sample):
    df = sheets[sheet_name]
    result = df.isin([sample])
    if result.any().any():
        # Dùng 'stack' để tìm tất cả vị trí của giá trị trong DataFrame
        positions = list(zip(*np.where(result)))
        
        # Hiển thị tất cả các vị trí tìm thấy
        for row, col in positions:
            print(f"Value found at row {row}, column {df.columns[col]}")
    else:
        print("Value not found in the table.")
    return int(row), int(col)

In [6]:
# assert find_sample(lst_sheets[0],'27032024-BOD-15-10-Q=49.66mL/phút-1') == (0,4)

In [7]:
# Re-define the function to process sheets based on the observed structure in the image
def process_sheet(sheet_name):
    df_sheet = pd.DataFrame(columns=['Loại', 'Doin (mV)', 'No.peak', 'DOmin (mV)', 'DDO (mV)','Tên sheet', 'Tên mẫu'])
    _,relevant_data = read_data(file_path,sheet_name)
    relevant_data = relevant_data.fillna(" ").iloc[:,1:]
    lst_samples=[]
    # Loop through the rows of the sheet
    for idx, row in relevant_data.iterrows():
        # Check for the presence of a sample name in the row (based on pattern in columns like 'U43-H1-VS2...')
        for col in row:
            if isinstance(col, str) and 'Q' in col:  # Pattern to match sample names
                lst_samples.append(col)

    for sample in lst_samples:
        _, cols = find_sample(sheet_name,sample)
        # Get the table in sheet base on sample name's column 
        min_col=cols - 1
        max_col=cols + 3
        dct = {}
        len_data = 0
        for col in range(max_col,min_col-1, -1):
            lst = []
            if col == min_col:
                start_row = relevant_data[col].loc[relevant_data[col].str.strip() != ""].first_valid_index()
                num_rows = len_data
                subset = relevant_data[col].iloc[start_row:start_row+num_rows].replace(" ", np.nan).ffill()
                lst =  subset.tolist()
            for value in relevant_data[col]:
                if not (isinstance(value, float) or isinstance(value, int)): 
                    pass
                else:
                    lst.append(value)
            dct[col]=lst
            len_data = len(lst)

        sorted_dct = dict(sorted(dct.items()))
        target_names = ['Loại', 'Doin (mV)', 'No.peak', 'DOmin (mV)', 'DDO (mV)']
        keys = list(sorted_dct.keys())
        i=0
        for key in keys:
            sorted_dct[target_names[i]]= sorted_dct.pop(key)
            i+=1

        df_sample = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in sorted_dct.items()]))
        df_sample['Tên sheet']= sheet_name
        df_sample['Tên mẫu']= sample

        # Loại bỏ các cột chứa toàn bộ NaN
        # df_sample = df_sample.dropna().infer_objects()
        # df_sheet = df_sheet.dropna()
        # Append the extracted information to the consolidated data
        df_sheet = pd.concat([df_sheet.astype(df_sample.dtypes), df_sample.astype(df_sheet.dtypes)], ignore_index=True)
        print(f"Sheet {sheet_name}\nMẫu {sample}")
    
    return df_sheet


In [8]:
# Initialize an empty DataFrame to hold the consolidated data
consolidated_data = pd.DataFrame(columns=['Loại', 'Doin (mV)', 'No.peak', 'DOmin (mV)', 'DDO (mV)','Tên sheet', 'Tên mẫu'])

# Process each sheet again with the updated logic
for sheet in lst_sheets:
    df_sheet = process_sheet(sheet)
    # consolidated_data = consolidated_data.dropna()
    consolidated_data = pd.concat([consolidated_data.astype(df_sheet.dtypes), df_sheet.astype(consolidated_data.dtypes)], ignore_index=True)
    
# Show the updated consolidated data
consolidated_data


Value found at row 0, column 3
Sheet U1-26.03.2024
Mẫu U1-GGA10-Zn(II) 5mg/L-17042024-Q=49.31mL/phút-2
Value found at row 0, column 3
Sheet U2-03.04.2024
Mẫu U2-BOD10-Zn(II) 10mg/L-10042024-Q=49.32mL/phút-1
Value found at row 0, column 9
Sheet U2-03.04.2024
Mẫu U2-BOD10-Zn(II) 5mg/L-11042024-Q=50.24mL/phút-1
Value found at row 0, column 15
Sheet U2-03.04.2024
Mẫu U2-BOD10-Ni(II) 10mg/L-11042024-Q=49.32mL/phút-1
Value found at row 0, column 21
Sheet U2-03.04.2024
Mẫu U2-BOD10-Ni(II) 5mg/L-12042024-Q=50.57mL/phút-1
Value found at row 0, column 3
Sheet U3-12.04.2024
Mẫu U3-GGA10-Cr(VI) 5mg/L-15042024-Q=49.39mL/phút-1
Value found at row 0, column 10
Sheet U3-12.04.2024
Mẫu U3-GGA10-Ni(II) 15mg/L-16042024-Q=50.63mL/phút-1
Value found at row 0, column 3
Sheet U4-15.04.2024
Mẫu U4-GGA10-Cr(VI) 10mg/L-18042024-Q=50.60mL/phút-1
Value found at row 0, column 4
Sheet U5 - 17.04.2024
Mẫu U5-BOD10-Cr(VI) 15mg/L-19042024-Q=50.12mL/phút-1
Value found at row 0, column 10
Sheet U5 - 17.04.2024
Mẫu U5-BO

,Loại,Doin (mV),No.peak,DOmin (mV),DDO (mV),Tên sheet,Tên mẫu
0,BOD10,288.5761,301,283.82,4.75609,U1-26.03.2024,U1-GGA10-Zn(II) 5mg/L-17042024-Q=49.31mL/phút-2
1,BOD10,288.665,769,283.73,4.935,U1-26.03.2024,U1-GGA10-Zn(II) 5mg/L-17042024-Q=49.31mL/phút-2
2,BOD10,288.5954,1245,283.6,4.99541,U1-26.03.2024,U1-GGA10-Zn(II) 5mg/L-17042024-Q=49.31mL/phút-2
3,BOD10,288.5774,1721,283.54,5.03739,U1-26.03.2024,U1-GGA10-Zn(II) 5mg/L-17042024-Q=49.31mL/phút-2
4,BOD10,288.5435,2196,283.85,4.69348,U1-26.03.2024,U1-GGA10-Zn(II) 5mg/L-17042024-Q=49.31mL/phút-2
...,...,...,...,...,...,...,...
5398,GGA2.5-Zn-10,276.8237,5508,263.9,12.92372,U179-H1-VS2-28.08.2024,U179-H1-VS2-GGA2.5-Zn(II) 10mg-L-30082024-Q=50...
5399,GGA2.5-Zn-10,276.7897,5980,264.63,12.15971,U179-H1-VS2-28.08.2024,U179-H1-VS2-GGA2.5-Zn(II) 10mg-L-30082024-Q=50...
5400,GGA2.5-Zn-10,276.5418,6454,264.73,11.81175,U179-H1-VS2-28.08.2024,U179-H1-VS2-GGA2.5-Zn(II) 10mg-L-30082024-Q=50...
5401,GGA2.5-Zn-10,276.746,6929,264.75,11.996,U179-H1-VS2-28.08.2024,U179-H1-VS2-GGA2.5-Zn(II) 10mg-L-30082024-Q=50...


In [9]:
consolidated_data.to_csv('metadata-gga-metal.csv')